### Imports

In [ ]:
import os
import sys
sys.path.append("..")
import json
import torch
import numpy as np
import pandas as pd
import string
import itertools
from pathlib import Path
import matplotlib.pyplot as plt
from tempogen.functional_utils import (_torch_sin, _torch_cos,_torch_gelu, _torch_softplus,
                                       _torch_sigmoid, _torch_identity, _torch_sqrt)
from tempogen.temporal_random_generation import get_p_edge, get_funcs, get_z_distribution 
from simulation.simulation_tools import get_optimal_sim_XY, simulate
from simulation.simulation_metrics import run_detection_metrics
from utils import create_random_intervention_dict
from tempogen.temporal_scm import TempSCM

rng = np.random.default_rng()

def prind(di): print(json.dumps(di, sort_keys=False, indent=4))

### Data Generation

In [ ]:
# paths
par_dir = Path(os.getcwd()).parents[1].as_posix() 
COL_NAMES = list(string.ascii_uppercase) + ["".join(a) for a in list(itertools.permutations(list(string.ascii_uppercase), r=2))]

FILENAME = "synth_tscm"
STRUCT_FORM = "NL_1L-3L+-"
N_SCMS = 25
N_SAMPLES = 500
WARMUP_STEPS = 20
INTERV_PERCENT = 0.1

# the space for the # vars during random generation
n_vars_space = {
    "vars": list(range(5, 10)),
    "prob": [0.2] * 5 
}

# dynamic version of get_n_vars, able to favor specific size of graphs 
def get_n_vars(vars, prob):
    return rng.choice(vars, p=prob)

n_lags_space = {
    "lag": [1, 2, 3],
    "prob": [0.3, 0.35, 0.35]
}

# dynamic version of get_n_lags, able to favor specific size of graphs 

def get_n_lags(lag, prob):
    return rng.choice(lag, p=prob)

# the space for the edge probability during random generation 
# dynamic version, depending on the # vars & # lags, to keep large graphs sparser; based on preconfigured options.
def p_edge_space(n_vars, n_lags):
    total_edges = (n_vars ** 2) * n_lags
    if total_edges < 100:
        values = [3, 5, 7]
    elif total_edges < 200:
        values = [5, 7, 9]
    else:
        values = [9, 12, 15]

    values = [v / total_edges for v in values]
    weights = [0.6, 0.3, 0.1]

    return {"values": values, "weights": weights}

funcs_space = {
    "functions": [
        _torch_identity,
        _torch_sqrt,
        _torch_sigmoid,
        _torch_sin,
        _torch_cos,
        _torch_gelu,
        _torch_softplus,
    ],
    "weights": [0.15, 0.15, 0.2, 0.1, 0.15, 0.1, 0.15]
}

z_distribution_space = {
    "functions": [
        torch.distributions.Normal(0, 0.005),
        torch.distributions.Uniform(-0.15, 0.15)
    ],
    "weights": [0.7, 0.3]
}

### Run Experiments

In [ ]:
all_results = []
for ctr in range(N_SCMS):

    # sample a random TSCM (ground-truth causal model)
    n_vars = get_n_vars(**n_vars_space)
    n_lags = get_n_lags(**n_lags_space)
    p_edge = get_p_edge(**p_edge_space(n_vars, n_lags))

    scm_true = TempSCM(
        method="C",
        n_vars=n_vars,
        n_lags=n_lags,
        p_edge=p_edge,
        funcs=get_funcs(**funcs_space),
        z_distributions=get_z_distribution(**z_distribution_space)
    )

    # sample observational data (true data)
    obs_data_sim = scm_true.generate_time_series(n_samples=N_SAMPLES)

    while np.isinf(obs_data_sim.values).any():
        obs_data_sim = scm_true.generate_time_series(n_samples=N_SAMPLES)

    # learn optimal causal model with TCS & ACT (NOTE: C2STs compare real and simulated observational data)
    try:
        obs_res = get_optimal_sim_XY(
            true_data=obs_data_sim,
            CONFIGS=None,
            sparsity_penalty=True,
            bias_correction=False,
            verbose=True
        )

        optimal_config = obs_res["optimal_config"]
        obs_auc = obs_res["auc"]

        # create intervention targets
        interv_dict = create_random_intervention_dict(
            scm=scm_true,
            obs_data=obs_data_sim,
            n_samples=N_SAMPLES,
            warmup_steps=WARMUP_STEPS,
            interv_percent=INTERV_PERCENT,
            intervention_type="hard"
        )

        # generate simulated interventional data from the ground truth SCM
        print(f'Generating interventional data from the ground truth SCM {ctr}...')
        interv_data_true = scm_true.generate_time_series_interv(
            n_samples=N_SAMPLES,
            interventional_dict=interv_dict,
            intervention_type="hard"
        )

        # obtain the configuration of the optimal causal model 
        CONFIG = {
                    "cd": optimal_config["cd"], 
                    "fc": optimal_config["fc"],
                    "z": optimal_config["z"],
                    "o": optimal_config["o"],
                }

        _, fitted_scm, _, _ = simulate(
            true_data=obs_data_sim,
            cd_method=CONFIG["cd"]["cd_method"],
            cd_kwargs=CONFIG["cd"]["cd_kwargs"],
            pred_method=CONFIG["fc"]["pred_method"],
            pred_kwargs=CONFIG["fc"]["pred_kwargs"],
            o_approximation=CONFIG["o"],
            noise_approximation=CONFIG["z"],
            verbose=True
        )

        # generate simulated interventional data from the fitted TSCM
        print(f"Generating interventional data from the fitted TSCM {ctr}...")
        interv_data_sim = fitted_scm.generate_time_series_interv(
            n_samples=N_SAMPLES,
            interventional_dict=interv_dict,
            intervention_type="hard"
        )

        # NOTE: C2STs compare real and simulated interventional data
        interv_auc = run_detection_metrics(
            real=interv_data_true,
            synthetic=interv_data_sim
        )["auc"]

        all_results.append({
            "scm_id": int(ctr),
            "obs_auc": obs_auc,
            "interv_auc": interv_auc,
            "n_vars": int(n_vars),
            "n_lags": (n_lags)
        })

        # paths for data and TSCMs 
        Path(f"{par_dir}/data/cp_style/interventions/{FILENAME}_{STRUCT_FORM}/obs/").mkdir(parents=True, exist_ok=True)
        Path(f"{par_dir}/data/cp_style/interventions/{FILENAME}_{STRUCT_FORM}/interv/").mkdir(parents=True, exist_ok=True)
        Path(f"{par_dir}/data/cp_style/interventions/{FILENAME}_{STRUCT_FORM}/structure/").mkdir(parents=True, exist_ok=True)

        # save data and TSCMs
        obs_data_sim.to_csv(f"{par_dir}/data/cp_style/interventions/{FILENAME}_{STRUCT_FORM}/obs/cp_collection_data_{ctr}.csv",
                            index=False)
        interv_data_sim.to_csv(f"{par_dir}/data/cp_style/interventions/{FILENAME}_{STRUCT_FORM}/interv/cp_collection_data_{ctr}.csv",
                               index=False)

        torch.save(fitted_scm.causal_structure.causal_structure_cp,
                   f"{par_dir}/data/cp_style/interventions/{FILENAME}_{STRUCT_FORM}/structure/cp_collection_struct_{ctr}.pt")
        # save results
        save_dir = Path(f"{par_dir}/data/results/interventions/")
        save_dir.mkdir(parents=True, exist_ok=True)

    except Exception as e:
        print(e)

    results = {
        "obs_auc": float(obs_auc),
        "interv_auc": float(interv_auc),
        "n_vars": float(n_vars),
        "n_lags": float(n_lags)
    }

    #with open(save_dir / f"{FILENAME}_{STRUCT_FORM}_{ctr}.json", "w") as f:
    #    json.dump(results, f, indent=2)

    print(f"SCM {ctr}: Obs. C2ST AUC={results['obs_auc']}, Interv. C2ST AUC={results['interv_auc']}")

    all_results.append(results)

df_results = pd.DataFrame(all_results)

#mean_obs_auc = df_results["obs_auc"].mean()
#mean_interv_auc = df_results["interv_auc"].mean()

#agg_results = {
#    "mean_obs_auc": float(mean_obs_auc),
#    "mean_interv_auc": float(mean_interv_auc),
#    "n_scms": len(df_results)
#}

agg_save_dir = Path(f"{par_dir}/data/results/interventions/")
agg_save_dir.mkdir(parents=True, exist_ok=True)

#with open(agg_save_dir / f"{FILENAME}_{STRUCT_FORM}_aggregated.json", "w") as f:
#    json.dump(agg_results, f, indent=2)

df_results.to_csv(
    agg_save_dir / f"{FILENAME}_{STRUCT_FORM}_per_scm_confidence.csv",
    index=False
)

### Plotting

In [ ]:
df_plot = df_results.sort_values("scm_id")  # sort by SCM id

# ---- ADD: small horizontal jitter ----
jitter = 0.08
x = df_plot["scm_id"].values

plt.figure(figsize=(6, 3))

plt.scatter(
    x - jitter,
    df_plot["obs_auc"],
    marker="o",
    label="Obs. AUC"
)

plt.scatter(
    x + jitter,   
    df_plot["interv_auc"],
    marker="s",
    label="Interv. AUC"
)

for xi, y1, y2 in zip(x, df_plot["obs_auc"], df_plot["interv_auc"]):
    plt.plot(
        [xi, xi],
        [y1, y2],
        alpha=0.3
    )

plt.xlabel("TSCM instance")
plt.ylabel("C2ST AUC$_D$")
plt.xticks(ticks=np.arange(min(x), max(x) + 1, 1))
plt.ylim(0.45, 1.0)
plt.legend()
plt.tight_layout()
plt.title("Fidelity of interventions on C2ST AUC$_D$")

plt.show()
